# Schedule bugs on a real GPU: how often do they show, and what do the timing tails look like?

> Run on Kaggle (Tesla T4). Results: `results/msync_gpu_races*.json` and `results/msync_gpu_latency*.json`
> (plus `.samples.json.gz`). See `docs/megakernel_sync.md`, "Part 3 results".

1. **Races.** Every part-1 schedule program (`msync.ir.reference_programs`) and every mutant
   (`msync.mutate`) runs as one persistent Triton kernel (`msync.gpu`, one program per SM, counter waits
   with acquire atomics, signals with release atomics, bounded spins). Each one runs `reps` times with no
   delays and with random injected delays. Each launch is compared with the step machine's reference
   output. Then, per oracle label: how many mutants did a GPU run ever expose, and how often?
2. **Latency.** Each benign reference runs as a static megakernel, as a dynamic (atomic work queue)
   megakernel, and as one launch per task, 100k launches each. Each series goes through
   [tinytail](https://github.com/rajul-kk/tinytail)'s i.i.d. gates and its EVT/pWCET fit.

3. **The same, on the CUDA C++ VM** (`msync.cuda_vm`, `csrc/msync_vm.h`). It has PTX acquire/release,
   `%globaltimer` timeouts, `__nanosleep` delays, the warp-level `no_barrier` mutant (labelled `unmodelled`)
   and an in-kernel latency series with no launch overhead.

In [ ]:
# Kaggle: Settings -> Accelerator -> GPU T4. Internet must be on.
import glob, os, shutil, subprocess, sys
os.environ["TRITON_INTERPRET"] = "0"
found = glob.glob("/kaggle/input/**/triton_equiv", recursive=True)
if found:   # a private dataset holding src/triton_equiv at a given commit (see SOURCE.txt)
    DS = found[0].rsplit("/", 1)[0]
    print("dataset at", DS, open(DS + "/SOURCE.txt").read() if os.path.exists(DS + "/SOURCE.txt") else "")
    shutil.copytree(DS, "/kaggle/working/tileproof_src", dirs_exist_ok=True)
else:
    subprocess.run(["git", "clone", "-q", "https://github.com/rajul-kk/triton_tileproof.git", "/kaggle/working/tp"], check=True)
    shutil.copytree("/kaggle/working/tp/src", "/kaggle/working/tileproof_src", dirs_exist_ok=True)
    shutil.copy("/kaggle/working/tp/results/labels.json", "/kaggle/working/labels.json")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "z3-solver", "ninja",
                "git+https://github.com/rajul-kk/tinytail.git"], check=True)
sys.path[:0] = ["/kaggle/working/tileproof_src"]

# newer statsmodels no longer re-exports InterpolationWarning from tsa.stattools, where tinytail imports it
import statsmodels.tsa.stattools as _st
if not hasattr(_st, "InterpolationWarning"):
    from statsmodels.tools.sm_exceptions import InterpolationWarning as _iw
    _st.InterpolationWarning = _iw


In [ ]:
import json, torch, triton
print(torch.__version__, triton.__version__, torch.cuda.get_device_name(0))
from triton_equiv.msync import gpu, races, latency
from triton_equiv.msync.ir import reference_programs
from triton_equiv.msync.machine import num_arena, num_inputs, run

## Control: every reference completes and matches, with and without delays

If one does not, stop here: the kernel's ordering is wrong, or `max_spin` is too small for the delays.
`moe_dispatch` is racy by design (ticket order), so only its outcome is checked.

In [ ]:
import random
for ref in reference_programs():
    arena = num_arena(ref, num_inputs(ref, 0), "nan")
    status, m = run(ref, arena)
    r = gpu.Runner(gpu.encode(ref), arena, "parallel")
    rng = random.Random(0)
    for k in range(20):
        d = None if k % 2 == 0 else races.random_delays(rng, len(ref.tasks), 1000)
        o = races.observe(r.run(d, races.max_spin_for(ref, 1000)), list(m.output()))
        assert o is races.Obs.MATCH or (ref.name == "moe_dispatch" and o is races.Obs.DIFFERS), (ref.name, k, o)
    print(ref.name, "ok")

## 1. Races: every mutant, 100 launches per condition

In [ ]:
res = races.main(["--reps", "100", "--dmax", "1000", "--out", "/kaggle/working/msync_gpu_races.json"])

In [ ]:
import pandas as pd
rows = [{"id": r["id"], "label": r["label"], "operator": r["operator"],
         **{f"{c}:{o}": r["obs"][c][o] for c in races.CONDITIONS for o in ("match", "differs", "timed_out", "faulted")}}
        for r in res["rows"]]
df = pd.DataFrame(rows)
pd.set_option("display.max_rows", 200)
df.sort_values(["label", "id"])

## 2. Latency and tails: 100k launches per (program, schedule)

In [ ]:
lat = latency.main(["--n", "100000", "--out", "/kaggle/working/msync_gpu_latency.json"])
pd.DataFrame([{k: r[k] for k in ("program", "tasks", "schedule", "median_us", "gate_verdict", "primary_outcome")}
              for r in lat["rows"]])

## 3. The CUDA C++ backend

Builds `csrc/msync_vm.cu` with `load_inline` (nvcc, sm_75). Same control as above first: every reference
completes and matches with and without delays, now in nanoseconds.

In [ ]:
from triton_equiv.cuda_ref import Backend
from triton_equiv.msync import cuda_vm
print(cuda_vm.build_error(Backend.CUDA) or "built")
for ref in reference_programs():
    arena = num_arena(ref, num_inputs(ref, 0), "nan")
    status, m = run(ref, arena)
    r = cuda_vm.CudaRunner(gpu.encode(ref), arena, "parallel", backend=Backend.CUDA)
    rng = random.Random(0)
    for k in range(20):
        d = None if k % 2 == 0 else races.random_delays(rng, len(ref.tasks), 20_000)
        o = races.observe(r.run(d, races.max_spin_for(ref, 20_000, 2_000_000)).launch, list(m.output()))
        assert o is races.Obs.MATCH or (ref.name == "moe_dispatch" and o is races.Obs.DIFFERS), (ref.name, k, o)
    print(ref.name, "ok")

In [ ]:
res_cuda = races.main(["--backend", "cuda", "--reps", "100", "--out", "/kaggle/working/msync_gpu_races_cuda.json"])

In [ ]:
lat_cuda = latency.main(["--backend", "cuda", "--n", "100000", "--out", "/kaggle/working/msync_gpu_latency_cuda.json"])
pd.DataFrame([{k: r[k] for k in ("program", "tasks", "schedule", "median_us", "gate_verdict", "primary_outcome")}
              for r in lat_cuda["rows"]])